# Olist — Causal EDA (Phase 1 appendix)

Exploratory analysis supporting the causal claim that shortening
`processing_time_days` (order_approved → order_delivered_carrier) **causes** a
reduction in `delay_days` (actual vs estimated delivery).

This notebook produces the figures and tables referenced in the appendix.

## Contents
1. Load cleaned per-order panel via `app.causal.dataset.load_order_features`.
2. Marginal distributions of treatment, outcome, confounders.
3. Naive correlation (biased).
4. Causal ATE via DoWhy backdoor adjustment (linear regression).
5. Placebo refuter (robustness).
6. Counterfactual sensitivity sweep over Δ.

Run order: execute cells top-to-bottom. The first DoWhy call takes ~30s.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from app.causal.dataset import load_order_features
from app.causal.effects import estimate_ate, estimate_counterfactual
from app.causal.graph import CONFOUNDERS, OUTCOME, TREATMENT

plt.rcParams['figure.figsize'] = (8, 4)
pd.set_option('display.max_columns', 20)

## 1. Load & describe

In [ ]:
df = load_order_features()
print('rows:', len(df))
df.head()

In [ ]:
df[[TREATMENT, OUTCOME, 'shipping_distance_km', 'product_weight_kg']].describe()

## 2. Marginal distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
df[TREATMENT].clip(-5, 30).hist(bins=60, ax=axes[0])
axes[0].set_title(f'{TREATMENT} (clipped)')
df[OUTCOME].clip(-40, 40).hist(bins=60, ax=axes[1])
axes[1].set_title(f'{OUTCOME} (clipped)')
plt.tight_layout()

## 3. Naive correlation (biased)

Shows the raw Pearson correlation between treatment and outcome — this does
**not** isolate the causal effect because of confounding by shipping
distance, weight, state, and seasonality.

In [ ]:
corr = df[[TREATMENT, OUTCOME]].corr().iloc[0, 1]
print(f'Pearson corr({TREATMENT}, {OUTCOME}) = {corr:+.4f}')

## 4. Causal ATE via DoWhy

Identifies the effect via the backdoor adjustment set
`{shipping_distance_km, product_weight_kg, seller_state, customer_state,
order_month}` declared in `app.causal.graph`, then estimates it by OLS.

We use a random 10k subsample for notebook speed; the full-data estimate is
exposed via `POST /causal/estimate-effect`.

In [ ]:
sample = df.sample(n=10000, random_state=42)
result = estimate_ate(sample, refute=False)
result

**Reading the output**

`ate` is the expected change in `delay_days` per +1 unit of
`processing_time_days`, holding the confounders fixed. If it is positive and
its CI excludes 0, we have evidence that reducing processing time causes a
reduction in delay.

## 5. Placebo refuter

Permutes the treatment and re-estimates — a robust result should drop to
approximately zero. Runs slower (~60s).

In [ ]:
refuted = estimate_ate(sample, refute=True)
print('ATE  :', refuted['ate'])
print('ATE CI:', refuted['ate_ci_lower'], refuted['ate_ci_upper'])
print('Placebo refutation p-value:', refuted['refutation_p_value'])

## 6. Counterfactual sensitivity sweep

For Δ ∈ {-3, -2, -1, 0, 1, 2}, compute the expected shift in `delay_days`.

In [ ]:
deltas = np.array([-3, -2, -1, 0, 1, 2])
shifts = []
for d in deltas:
    r = estimate_counterfactual(sample, intervention_delta=float(d))
    shifts.append(r['expected_outcome_shift'])

pd.DataFrame({'delta_days': deltas, 'expected_delay_shift': shifts})

In [ ]:
plt.plot(deltas, shifts, 'o-')
plt.axhline(0, color='grey', lw=0.5)
plt.axvline(0, color='grey', lw=0.5)
plt.xlabel('Intervention on processing_time_days (Δ)')
plt.ylabel('Expected shift in delay_days')
plt.title('Counterfactual sensitivity — Olist (causal)')
plt.grid(alpha=0.3)

---
_All runs in this notebook are automatically logged to MLflow (experiment
`causal`). That run table is the audit trail._